# PornWorks Web Generator — Image quality fix

This notebook launches a prompt-to-image website using the PornWorks Real Porn V03 SDXL model. This revision keeps the SDXL VAE in float32 to reduce black, noisy, or pixel-corrupted outputs that can result from VAE precision issues.

**Start clean:** Run the setup cell, wait for it to finish, then select **Runtime → Restart session**. Run all remaining cells in order. Choose **Runtime → Change runtime type → T4 GPU** if available.

The Gradio share link is temporary. Use only fictional consenting adults and do not create intimate imagery of real people without their explicit consent.

In [ ]:
# Remove the incompatible optional quantizer and replace Diffusers/Transformers with pinned compatible versions
!pip uninstall -y torchao diffusers transformers
!pip -q install "diffusers==0.32.2" "transformers==4.48.3" "huggingface-hub<1.0" accelerate safetensors gradio

In [ ]:
import importlib.util
if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError("torchao is still installed. Run the setup cell again, then Runtime → Restart session.")

import torch
import gradio as gr
from diffusers import DiffusionPipeline

MODEL_ID = "John6666/pornworks-real-porn-v03-sdxl"
has_cuda = torch.cuda.is_available()
dtype = torch.float16 if has_cuda else torch.float32

print("GPU available:", has_cuda)
if has_cuda:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. CPU generation will be extremely slow; select a GPU runtime if possible.")

pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype, use_safetensors=True)
# Keep the SDXL VAE in float32 to avoid precision-related decode artifacts.
pipe.vae.to(dtype=torch.float32)
if has_cuda:
    pipe.enable_model_cpu_offload()
    pipe.enable_attention_slicing()
else:
    pipe.to("cpu")
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print("Model loaded. VAE dtype:", next(pipe.vae.parameters()).dtype)

In [ ]:
import random
import numpy as np

def generate_image(prompt, negative_prompt, width, height, steps, guidance, seed):
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("Enter a prompt first.")
    if width % 8 or height % 8:
        raise gr.Error("Width and height must be divisible by 8.")
    actual_seed = int(seed)
    if actual_seed < 0:
        actual_seed = random.randint(0, 2**31 - 1)
    generator_device = "cuda" if has_cuda else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(actual_seed)
    try:
        with torch.inference_mode():
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt or None,
                width=int(width),
                height=int(height),
                num_inference_steps=int(steps),
                guidance_scale=float(guidance),
                generator=generator,
            )
        image = result.images[0].convert("RGB")
        arr = np.asarray(image)
        if arr.size == 0 or not np.isfinite(arr).all() or arr.std() < 1.0:
            raise gr.Error("The model returned an invalid or nearly blank image. Restart the Colab runtime and try again.")
        return image, f"Seed: {actual_seed} | Size: {image.width}×{image.height}"
    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Generation failed: {type(e).__name__}: {e}")

custom_css = """
body { background: #101014; }
.gradio-container { max-width: 1100px !important; }
#hero { text-align: center; padding: 18px 0 8px; }
#hero h1 { letter-spacing: -1px; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="violet", neutral_hue="slate"), css=custom_css, title="PornWorks Image Lab") as demo:
    gr.Markdown("# PornWorks Image Lab\nCreate photorealistic images with the Real Porn V03 SDXL model. Use fictional consenting adults only.", elem_id="hero")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", placeholder="Describe the fictional adult characters, scene, lighting, camera, and style…", lines=5)
            negative = gr.Textbox(label="Negative prompt", value="pixelated, low quality, blurry, jpeg artifacts, distorted anatomy, extra fingers, watermark, text", lines=2)
            with gr.Accordion("Generation settings", open=True):
                width = gr.Slider(512, 1024, value=768, step=64, label="Width")
                height = gr.Slider(512, 1024, value=768, step=64, label="Height")
                steps = gr.Slider(20, 50, value=35, step=1, label="Steps")
                guidance = gr.Slider(1, 10, value=5.0, step=0.5, label="Guidance scale")
                seed = gr.Number(value=-1, precision=0, label="Seed (-1 = random)")
            run = gr.Button("Generate image", variant="primary")
        with gr.Column(scale=1):
            output = gr.Image(label="Generated image", type="pil", format="png")
            seed_output = gr.Textbox(label="Generation info", interactive=False)
    run.click(generate_image, inputs=[prompt, negative, width, height, steps, guidance, seed], outputs=[output, seed_output])
    gr.Markdown("**Privacy note:** This prototype does not intentionally save generated images to a database, but prompts and outputs are processed by the hosted runtime. Do not enter personal or confidential information.")

demo.queue(max_size=2).launch(share=True, debug=True)

## How to use
1. Run the setup cell and wait until it finishes.
2. Select **Runtime → Restart session**.
3. Run the remaining cells from top to bottom.
4. Wait for the model to load, then open the `gradio.live` link printed by the final cell.
5. Generate a test image at 768×768 first. Keep the Colab runtime running while using the website.

This revision keeps the SDXL VAE in float32 to reduce decode artifacts. If the browser shows an XML `AccessDenied` page, return to the live Gradio app link printed by the final cell rather than opening an old temporary image URL; share links and output URLs can expire when the runtime stops.